### 1\. Configuração Inicial (Importações e Dados)

Antes de tudo, precisamos importar todas as bibliotecas que usaremos (como `pandas`, `sklearn`, etc.). Também é aqui que carregaríamos nossos dados (por exemplo, de um arquivo CSV) para o DataFrame `df`.

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

df_0 = pd.read_excel("https://github.com/2025-2-NCC5/Projeto6/raw/main/documentos/base_cannoli.xlsx")

### 2\. Criando subcolunas decendentes das atuais

Criando colunas que possam auxiliar a análise a partir das colunas existentes.

In [2]:
# Tratar valores nulos
df = df_0.fillna({
    'totalAmount': 0
})

# Padronizar datas
for col in df.columns:
    if any(x in col.lower() for x in ['sendat', 'purchasedat', 'dateofbirth']):
        df[col] = pd.to_datetime(df[col], errors='coerce')

# Criar features temporais
# 1 - Campanha
if 'sendAt' in df.columns:
    df['sendAt_month'] = df['sendAt'].dt.month
    df['sendAt_week'] = df['sendAt'].dt.isocalendar().week
    df['sendAt_weekday_name'] = df['sendAt'].dt.day_name()

# 2 - Compra
if 'purchasedAt' in df.columns:
    df['purchasedAt_month'] = df['purchasedAt'].dt.month
    df['purchasedAt_week'] = df['purchasedAt'].dt.isocalendar().week
    df['purchasedAt_weekday_name'] = df['purchasedAt'].dt.day_name()

# 3 - Idade
if 'dateOfBirth' in df.columns:
    today = pd.to_datetime('today').normalize()
    df['age'] = np.floor((today - df['dateOfBirth']).dt.days / 365.25)

# 4 - Cliente impactado pela campanha
if {'sendAt', 'purchasedAt'}.issubset(df.columns):
    df['clienteImpactado'] = np.where(
        (df['purchasedAt'] >= df['sendAt']) &
        (df['purchasedAt'] <= df['sendAt'] + pd.Timedelta(days=7)),
        1, 0
    )
else:
    df['clienteImpactado'] = 0  # caso não existam as colunas

# Codificar apenas colunas específicas
cat_cols = ['gender', 'salesChannel']
encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

# Exibindo os resultados da transformação ("Dicionário" das variáveis)
for col, le in encoders.items():
    print(f"\nColuna: {col}")
    for i, classe in enumerate(le.classes_):
        print(f"  {i} → {classe}")

# Resultados
print("\n✅ Tratamento concluído!")
df.info()


Coluna: gender
  0 → N
  1 → O

Coluna: salesChannel
  0 → 99FOOD
  1 → ANOTAAI
  2 → APP
  3 → IFOOD
  4 → WEB
  5 → WHATSAPP
  6 → nan

✅ Tratamento concluído!
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 21 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   customerid                5000 non-null   object        
 1   name                      5000 non-null   object        
 2   gender                    5000 non-null   int64         
 3   contrycode                5000 non-null   int64         
 4   areacode                  5000 non-null   int64         
 5   phonenumber               5000 non-null   int64         
 6   campaignName              3975 non-null   object        
 7   response                  1768 non-null   object        
 8   sendAt                    5000 non-null   datetime64[ns]
 9   purchasedAt               4021 non-null   d

-----

### 3\. Preparação de Dados (Feature Engineering)

Nesta etapa, criamos nossa variável alvo (o que queremos prever) e limpamos os dados.

1.  **`target_comprou`**: Transformamos o problema em uma classificação binária (0 ou 1). Se o `totalAmount` for maior que zero, o cliente comprou (`1`), senão, não comprou (`0`).
2.  **Tratar NaNs**: Preenchemos valores ausentes (NaNs) nas colunas categóricas com valores padrão (como 'Nenhuma' ou 'Desconhecido') para que o modelo possa processá-los.

<!-- end list -->

In [3]:
# Criar a variável Alvo
# target_comprou é 1 APENAS SE totalAmount > 0
df['target_comprou'] = (df['totalAmount'] > 0).astype(int)

# Tratar NaNs nas features categóricas
df['campaignName'] = df['campaignName'].fillna('Nenhuma')
df['response'] = df['response'].fillna('Nao_Respondeu')
df['salesChannel'] = df['salesChannel'].fillna('Desconhecido')

-----

### 4\. Definição de Features (X) e Alvo (y)

Agora, separamos formalmente o nosso conjunto de dados em "features" (`X`), que são as variáveis de entrada que o modelo usará para aprender, e "alvo" (`y`), que é a variável que queremos prever.

  * **`numeric_features`**: Lista de colunas que contêm números (idade, dia da semana).
  * **`categorical_features`**: Lista de colunas que contêm texto/categorias (gênero, país).
  * **`X`**: DataFrame contendo apenas as colunas de features.
  * **`y`**: Series (coluna) contendo apenas o alvo.

<!-- end list -->

In [4]:
# Definindo as features numéricas
numeric_features = ['age']

# Definindo as variaveis categóricas
categorical_features = [
    'gender',
    'contrycode',
    'areacode',
    'campaignName',
    'response',
    'sendAt_weekday_name'
]

features = numeric_features + categorical_features
target = 'target_comprou'

X = df[features]
y = df[target]

-----

### 5\. Criação do Pipeline de Pré-processamento

Modelos de machine learning não entendem texto ou valores ausentes. Esta é a etapa mais importante da preparação. Criamos "pipelines" para tratar automaticamente as colunas numéricas e categóricas.

In [5]:
# Tratando variáveis numéricas
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

# Tratando variáveis categóricas
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combinando os pipelines
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

-----

### 6\. Definição dos Modelos e do Ensemble (VotingClassifier)

Em vez de confiar em um único modelo, vamos usar um "comitê" de modelos, chamado `VotingClassifier`. Isso geralmente leva a previsões mais robustas.

In [6]:
# Modelo 1: Regressão Logística
clf1 = LogisticRegression(random_state=42, max_iter=1000)

# Modelo 2: Random Forest
clf2 = RandomForestClassifier(random_state=42)

# Modelo 3: Support Vector Machine (SVC)
clf3 = SVC(probability=True, random_state=42)

# Modelo 4: K-Nearest Neighbors (KNN)
clf4 = KNeighborsClassifier()

# Criando o VotingClassifier
eclf1 = VotingClassifier(
    estimators=[
        ('lr', clf1),
        ('rf', clf2),
        ('svc', clf3),
        ('knn', clf4)
    ],
    voting='soft'
)

# --- Criando o Pipeline FINAL com SMOTE ---
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)), # Etapa de balanceamento
    ('classifier', eclf1)
])

-----

### 7\. Treinamento e Avaliação do Modelo

Com o pipeline principal pronto, agora podemos treinar e avaliar.

In [7]:
# --- 6. Treinamento e Avaliação (com SMOTE) ---

# Divisão em Treino e Teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Treinando o modelo
model_pipeline.fit(X_train, y_train)

# Fazendo previsões
y_pred = model_pipeline.predict(X_test)

# Avaliando o modelo
print("\n--- Avaliação do VotingClassifier ---")
print(f"Acurácia: {accuracy_score(y_test, y_pred):.4f}")
print("\nRelatório de Classificação:")
print(classification_report(y_test, y_pred, target_names=['Nao Comprou', 'Comprou']))


--- Avaliação do VotingClassifier ---
Acurácia: 0.6147

Relatório de Classificação:
              precision    recall  f1-score   support

 Nao Comprou       0.19      0.31      0.24       294
     Comprou       0.80      0.69      0.74      1206

    accuracy                           0.61      1500
   macro avg       0.50      0.50      0.49      1500
weighted avg       0.68      0.61      0.64      1500



### Análise dos Resultados e Próximo Passo

O último resultado do nosso `VotingClassifier` (combinado com o `SMOTE`) é o primeiro que podemos considerar de um modelo "real".

```
--- Avaliação do VotingClassifier ---
Acurácia: 0.6147

Relatório de Classificação:
              precision    recall  f1-score   support

 Nao Comprou       0.19      0.31      0.24       294
     Comprou       0.80      0.69      0.74      1206

    accuracy                           0.61      1500
   macro avg       0.50      0.50      0.49      1500
weighted avg       0.68      0.61      0.64      1500
```

#### Interpretação

1.  **Acurácia de 61% é "Honesta":** Diferente dos 80% anteriores (que eram uma farsa, apenas a proporção da classe majoritária), esta acurácia reflete um modelo que está *tentando* classificar ambas as classes.
2.  **O SMOTE Funcionou:** O `Recall` da classe `Nao Comprou` subiu de 0.00 para 0.31. Isso prova que o modelo foi forçado a parar de ignorar a classe minoritária e conseguiu identificar 31% dela.
3.  **O Modelo é Fraco:** Apesar de "funcionar", o desempenho geral é baixo. Um `f1-score` de 0.24 para a classe `Nao Comprou` é ruim. O modelo tem baixa precisão (0.19) e baixo recall (0.31), o que significa que ele erra muito ao tentar encontrar os não-compradores.

**Conclusão:** Nossa arquitetura atual (Pipeline com SMOTE + VotingClassifier com 4 modelos) é **muito complexa** e está entregando **pouco resultado**.
Com isso, realizaremos um teste utilizando somente o modelo XBoost, a fim de entender quais as principais variáveis que estão fazendo os clientes comprarem, juntamente com um modelo de regressão linear para quantificar esse impacto (nas variáveis numéricas não categóricas)

In [8]:
clf_xgb = XGBClassifier(
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss',
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum()
)

# Criando o Pipeline final
model_pipeline_xgb = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', clf_xgb)
])

In [9]:
# Divisão em Treino e Teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Treinando o modelo
model_pipeline_xgb.fit(X_train, y_train)

# Fazendo previsões
y_pred_rf = model_pipeline_xgb.predict(X_test)

# Avaliando o modelo
print("\n--- Avaliação do Random Forest (com class_weight='balanced') ---")
print(f"Acurácia: {accuracy_score(y_test, y_pred_rf):.4f}")
print("\nRelatório de Classificação:")
print(classification_report(y_test, y_pred_rf, target_names=['Nao Comprou', 'Comprou']))

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [16:18:26] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



--- Avaliação do Random Forest (com class_weight='balanced') ---
Acurácia: 0.6147

Relatório de Classificação:
              precision    recall  f1-score   support

 Nao Comprou       0.19      0.30      0.23       294
     Comprou       0.80      0.69      0.74      1206

    accuracy                           0.61      1500
   macro avg       0.50      0.49      0.49      1500
weighted avg       0.68      0.61      0.64      1500



In [ ]:
print("\n--- Extraindo Insights (Usando XGBoost) ---")

try:
    # Pega o 'preprocessor' do pipeline do XGBoost
    preprocessor_xgb = model_pipeline_xgb.named_steps['preprocessor']

    # Pega o modelo 'classifier' (o XGBoost) do pipeline
    xgb_model = model_pipeline_xgb.named_steps['classifier']

    # Nomes das features categóricas após o OneHotEncoding
    cat_feature_names = preprocessor_xgb.named_transformers_['cat'] \
        .named_steps['onehot'] \
        .get_feature_names_out(categorical_features)

    # Todos os nomes (Numéricas + Categóricas)
    all_feature_names = numeric_features + list(cat_feature_names)

    # Importâncias do modelo XGBoost
    importances = xgb_model.feature_importances_

    # Criando o DataFrame de insights
    insights_df = pd.DataFrame({
        'Feature': all_feature_names,
        'Importance': importances
    }).sort_values(by='Importance', ascending=False)

    print("--- Principais Variáveis que influenciam a Compra (Segundo o XGBoost) ---")
    print(insights_df.head(10))

except Exception as e:
    print(f"Erro ao extrair feature importances: {e}")


--- Extraindo Insights (Usando XGBoost) ---
--- Principais Variáveis que influenciam a Compra (Segundo o XGBoost) ---
                                          Feature  Importance
15    campaignName_Recompensa para Clientes Fieis    0.055013
1245                   sendAt_weekday_name_Monday    0.054413
1248                 sendAt_weekday_name_Thursday    0.050544
12              campaignName_Nova Carta de Drinks    0.049674
11                           campaignName_Nenhuma    0.047939
1247                   sendAt_weekday_name_Sunday    0.047818
13                   campaignName_Promover Almoço    0.047484
1244                   sendAt_weekday_name_Friday    0.047394
16           campaignName_Semana do Café Especial    0.046832
9               campaignName_Happy Hour Estendido    0.046746


Análise de Importância das Features (Modelo XGBoost) -
Esta análise é baseada no XGBClassifier, o modelo que apresentou o desempenho mais robusto em nosso processo, alcançando o melhor f1-score (0.24) e Recall (0.32) para a classe minoritária (Nao Comprou). Portanto, estes insights são os mais confiáveis.

Principais Observações:

Distribuição de Importância Plana: Não há uma única feature dominante. A importância é distribuída de forma muito uniforme, com o 1º lugar (5.5%) sendo marginalmente mais importante que o 10º (4.6%). Isso indica que nenhum atributo único tem alto poder preditivo.

Foco em Campanhas e Timing: O modelo identifica que os únicos preditores úteis nos dados atuais são o tipo de campanha (Recompensa..., Nova Carta...) e o dia de envio (Monday, Thursday).

Significado de campaignName_Nenhuma: O fato de "não estar em uma campanha" (4.7%) ser tão relevante quanto uma campanha ativa é um insight importante, mostrando que a ausência de intervenção é um preditor válido.

Conclusão Estratégica:

O desempenho limitado do modelo (f1-score de 0.24) é um reflexo direto desta lista. O XGBoost está combinando múltiplos sinais fracos para tomar uma decisão.